In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
ingr = pd.read_csv('/content/recipe_ingredients.csv')

In [ ]:
ingr

,recipe_id,ingredient,quantity,unit
0,1,картофель,1,кг
1,1,жир,40,г
2,1,соль,NaN,по вкусу
3,2,масло горчичный,1,г
4,2,хлеб чёрный,4-5,кусочков
...,...,...,...,...
1512,195,апельина,89-2,шт.
1513,195,яблоко,16-2,шт.
1514,195,майонез,8-4,ст. ложки
1515,195,с лимонный,59-2,ст. ложки


In [ ]:
rec = pd.read_csv("/content/recipes.csv")

In [ ]:
rec

,id,title,instructions,url,source,ingredients_text
0,1,Картофельные шарики во фритюре,Рецепт приготовления картофельных шариков.\nОч...,https://www.russianfood.com/recipes/recipe.php...,russianfood,картофель жир соль
1,2,Горячие бутерброды с сардельками,Ломти хлеба покрыть горчичным маслом. Сардельк...,https://www.russianfood.com/recipes/recipe.php...,russianfood,масло горчичный хлеб чёрный горчица сарделька ...
2,3,"Горячие бутерброды с яйцом, килькой и цветной...",Хлеб обжарить в масле до светло-желтого оттенк...,https://www.russianfood.com/recipes/recipe.php...,russianfood,хлеб белый пшеничный масло яйцо зелень петрушк...
3,4,Горячие бутерброды с копченой рыбой и яблоками,"Очищенную копченую рыбу размельчить, потушить ...",https://www.russianfood.com/recipes/recipe.php...,russianfood,хлеб белый пшеничный маргарин рыба копчёный му...
4,5,Сырные палочки с картофелем,Картофель отварить в подсоленной воде до мягко...,https://www.russianfood.com/recipes/recipe.php...,russianfood,картофель сыр яйцо масло мука соль перец чёрный
...,...,...,...,...,...,...
190,191,Десертный салат из курицы с ананасом и вином,"Шампинь хош прт, опуть касрлю подс кипящ вод...",https://www.russianfood.com/recipes/recipe.php...,russianfood,куиной ил ананас ерованна шона майонез вино сх...
191,192,Салат с курятиной и апельсинами,"Рис омыь, сит в у подсю (2 сакана), варь, т о...",https://www.russianfood.com/recipes/recipe.php...,russianfood,куица жар иликуица отный апельина р жёлтый му ...
192,193,Салат из курицы с апельсинами и бананами,"Куят нарь ш ку. Апельин исит от р, рь доль и ...",https://www.russianfood.com/recipes/recipe.php...,russianfood,майонез с консвира чеш банан апельина куица отный
193,194,Салат из курицы с мандаринами,NaN,https://www.russianfood.com/recipes/recipe.php...,russianfood,NaN


In [ ]:
!pip install sqlalchemy pandas

In [ ]:
recipes_df = pd.read_csv('/content/recipes.csv')
ingredients_df = pd.read_csv('/content/recipe_ingredients.csv')

In [ ]:
import os
if os.path.exists("recipes.db"):
    os.remove("recipes.db")

In [ ]:
from sqlalchemy.orm import declarative_base, relationship
from sqlalchemy import Column, Integer, String, ForeignKey, Text, Float

Base = declarative_base()


class Recipe(Base):
    __tablename__ = 'recipes'

    id = Column(Integer, primary_key=True)
    title = Column(String)
    instructions = Column(Text)
    url = Column(String)
    source = Column(String)
    ingredients_text = Column(Text)

    ingredient_ids = Column(Text)

    ingredients = relationship("Ingredient", back_populates="recipe")


class IngredientName(Base):
    __tablename__ = 'ingredient_names'

    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String, unique=True)


class Ingredient(Base):
    __tablename__ = 'ingredients'

    id = Column(Integer, primary_key=True, autoincrement=True)

    recipe_id = Column(Integer, ForeignKey('recipes.id'))
    ingredient_id = Column(Integer, ForeignKey('ingredient_names.id'))

    quantity = Column(String)
    unit = Column(String)

    recipe = relationship("Recipe", back_populates="ingredients")
    ingredient_name = relationship("IngredientName")

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///recipes.db')
Base.metadata.create_all(engine)

In [ ]:
from sqlalchemy.orm import sessionmaker

Session = sessionmaker(bind=engine)
session = Session()

In [ ]:
import pandas as pd

recipes_df = pd.read_csv('/content/recipes.csv')
ingredients_df = pd.read_csv('/content/recipe_ingredients.csv')

In [ ]:
unique_names = ingredients_df['ingredient'].dropna().unique()

name_to_id = {}

for name in unique_names:
    ing = IngredientName(name=name)
    session.add(ing)
    session.flush()  # получаем id сразу

    name_to_id[name] = ing.id

session.commit()

In [ ]:
for _, row in recipes_df.iterrows():
    recipe = Recipe(
        id=row['id'],
        title=row['title'],
        instructions=row['instructions'],
        url=row['url'],
        source=row['source'],
        ingredients_text=row['ingredients_text']
    )
    session.add(recipe)

session.commit()

In [ ]:
from collections import defaultdict

recipe_to_ingredient_ids = defaultdict(list)

for _, row in ingredients_df.iterrows():

    ing_name = row['ingredient']
    ing_id = name_to_id.get(ing_name)

    quantity = (row['quantity']) if pd.notna(row['quantity']) else None
    unit = row['unit'] if pd.notna(row['unit']) else None

    ingredient = Ingredient(
        recipe_id=row['recipe_id'],
        ingredient_id=ing_id,
        quantity=quantity,
        unit=unit
    )

    session.add(ingredient)

    recipe_to_ingredient_ids[row['recipe_id']].append(ing_id)

session.commit()

In [ ]:
for recipe_id, ing_ids in recipe_to_ingredient_ids.items():

    recipe = session.query(Recipe).get(recipe_id)

    recipe.ingredient_ids = ",".join(map(str, ing_ids))

session.commit()

/tmp/ipykernel_691/3368887245.py:3: LegacyAPIWarning: The Query.get() method is considered legacy as of the 1.x series of SQLAlchemy and becomes a legacy construct in 2.0. The method is now available as Session.get() (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  recipe = session.query(Recipe).get(recipe_id)


In [ ]:
recipes = session.query(Recipe).all()

for r in recipes[:5]:
    print(f"\n {r.title}")
    print("IDs:", r.ingredient_ids)

    for ing in r.ingredients:
        print(" -", ing.ingredient_name.name, ing.quantity, ing.unit)


📌 Сырные палочки с картофелем
IDs: 1,2,3,4,5,6,7
 - картофель 200.0 г
 - сыр 200.0 г
 - яйцо 3.0 шт.
 - масло 1.0 ст. ложка
 - мука 75.0 г
 - соль None по вкусу
 - перец чёрный None по вкусу
 - картофель 200 г
 - сыр 200 г
 - яйцо 3 шт.
 - масло 1 ст. ложка
 - мука 75 г
 - соль None по вкусу
 - перец чёрный None по вкусу

📌 Горячие бутерброды с копченой рыбой и яблоками
IDs: 8,9,10,11,12,13,14,15,6
 - хлеб белый пшеничный 4-6 кусочков
 - маргарин 50 г
 - рыба копчёный 250 г
 - мука пшеничный 1 ст. ложка
 - сметана 0.5 стакана
 - яблоко мочёный 2-3 шт.
 - лук 1 шт.
 - перец None по вкусу
 - соль None по вкусу

📌 Горячие бутерброды с яйцом, килькой  и цветной капустой
IDs: 8,4,3,16,17,18,19,6
 - хлеб белый пшеничный 4 кусочка
 - масло 20 г
 - яйцо 2 шт.
 - зелень петрушка 1 ст. ложка
 - килька 8-10 шт.
 - томатный паста 1-2 ст. ложки
 - капуста цветной 1 вилок
 - соль None по вкусу

📌 Картофельные шарики во фритюре
IDs: 1,20,6
 - картофель 1 кг
 - жир 40 г
 - соль None по вкусу

📌 Горяч